In [1]:
%pip install -q "flax>=0.12.2" "jax>=0.8.2" "chex>=0.1.91"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import jax
import jax.numpy as jnp
from flax import nnx

from ptrnet import network as net, ac

In [3]:
rngs = nnx.Rngs(0)

In [4]:
static = jax.random.uniform(rngs.default(), shape=(5, 2, 3))  # (B, M, N)
dynamic = jax.random.uniform(rngs.default(), shape=(5, 2, 3))  # (B, M, N)
x0 = static[:, :, :1]

print("Static:", static.shape)
print("Dynamic:", static.shape)

Static: (5, 2, 3)
Dynamic: (5, 2, 3)


In [5]:
ptrnet = net.PointerNetwork(rngs=rngs)
nnx.display(ptrnet)

In [6]:
initial_carry = ptrnet.init(batch_dim=x0.shape[0], rngs=rngs)

In [7]:
logits, carry = ptrnet(static, dynamic, x0, carry=initial_carry, rngs=rngs, training=False)
logits.shape

(5, 3)

In [8]:
actor = ac.Actor(ptrnet=ptrnet, greedy=True)

In [9]:
initial_carry = actor.init(batch_dim=x0.shape[0], rngs=rngs)

In [10]:
pointer, logprob, actprobs, carry = actor.action(
    static, dynamic, x0, carry=initial_carry, mask=None, rngs=rngs
)

In [11]:
pointer

Array([[2],
       [1],
       [1],
       [2],
       [0]], dtype=int32)